In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
import pandas as pd, json, random, torch
from pathlib import Path

In [3]:
random.seed(42)
print("GPU:", torch.cuda.get_device_name(0))
print("GPUs visible:", torch.cuda.device_count())

GPU: Tesla T4
GPUs visible: 1


In [4]:
df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

In [5]:
# Fill missing age with median, drop cabin, fill embarked
df['Age']      = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna('S')
df = df.drop(columns=['Cabin', 'Ticket', 'PassengerId'])

In [6]:
print("Shape:", df.shape)
print("Survived distribution:\n", df['Survived'].value_counts())
df.head(3)

Shape: (891, 9)
Survived distribution:
 Survived
0    549
1    342
Name: count, dtype: int64


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,7.2500,S
1,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,71.2833,C
2,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,7.9250,S


In [7]:
EMBARKED_MAP = {'S': 'Southampton', 'C': 'Cherbourg', 'Q': 'Queenstown'}
CLASS_MAP    = {1: 'first', 2: 'second', 3: 'third'}

In [8]:
QUESTIONS = [
    'Did this Titanic passenger survive?',
    'Based on this passenger profile, did they survive the Titanic?',
    'Given what we know about this passenger, predict their survival.',
]

In [9]:
def build_reasoning(row):
    name     = row['Name'].split(',')[0].strip()
    sex      = row['Sex']
    age      = int(row['Age'])
    pclass   = CLASS_MAP.get(row['Pclass'], str(row['Pclass']))
    fare     = float(row['Fare'])
    sibsp    = int(row['SibSp'])
    parch    = int(row['Parch'])
    embarked = EMBARKED_MAP.get(row['Embarked'], row['Embarked'])
    label    = 'Survived' if row['Survived'] == 1 else 'Did not survive'
    family   = sibsp + parch
 
    # Natural language reasoning paragraph
    age_note    = 'a child' if age < 15 else 'elderly' if age > 60 else 'an adult'
    class_note  = 'had priority access to lifeboats' if row['Pclass'] == 1 else \
                  'had moderate lifeboat access' if row['Pclass'] == 2 else \
                  'had limited lifeboat access'
    sex_note    = 'Women were prioritised in the evacuation.' if sex == 'female' else \
                  'Men had lower evacuation priority.'
    fare_note   = f'Paid a {"high" if fare > 50 else "moderate" if fare > 15 else "low"} fare of £{fare:.1f}.'
    family_note = f'Travelled with {family} family member(s).' if family > 0 else 'Travelled alone.'
 
    reasoning = (
        f"{name} was {age_note} {sex} passenger travelling {pclass} class, "
        f"boarding at {embarked}. {sex_note} "
        f"As a {pclass} class passenger, they {class_note}. "
        f"{fare_note} {family_note} "
        f"Prediction: {label}."
    )
    return reasoning

In [10]:
def row_to_example(row, question):
    facts = (
        f"  - Name: {row['Name']}\n"
        f"  - Sex: {row['Sex']}\n"
        f"  - Age: {int(row['Age'])}\n"
        f"  - Class: {row['Pclass']}\n"
        f"  - Fare: £{row['Fare']:.2f}\n"
        f"  - Siblings/Spouse aboard: {row['SibSp']}\n"
        f"  - Parents/Children aboard: {row['Parch']}\n"
        f"  - Embarked from: {EMBARKED_MAP.get(row['Embarked'], row['Embarked'])}"
    )
    return {
        'instruction': f"{question}\n\nPassenger profile:\n{facts}",
        'output': build_reasoning(row),
    }

In [11]:
# Generate — 3 variants per row
examples = []
for _, row in df.iterrows():
    for q in QUESTIONS:
        examples.append(row_to_example(row, q))

In [12]:
random.shuffle(examples)
print(f"Total examples: {len(examples)}")
print("\n--- Sample ---")
print(examples[0]['instruction'])
print()
print(examples[0]['output'])

Total examples: 2673

--- Sample ---
Given what we know about this passenger, predict their survival.

Passenger profile:
  - Name: O'Brien, Mr. Thomas
  - Sex: male
  - Age: 28
  - Class: 3
  - Fare: £15.50
  - Siblings/Spouse aboard: 1
  - Parents/Children aboard: 0
  - Embarked from: Queenstown

O'Brien was an adult male passenger travelling third class, boarding at Queenstown. Men had lower evacuation priority. As a third class passenger, they had limited lifeboat access. Paid a moderate fare of £15.5. Travelled with 1 family member(s). Prediction: Did not survive.


In [13]:
Path('output').mkdir(exist_ok=True)
with open('output/train.jsonl', 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex) + '\n')
print(f"Saved {len(examples)} examples.")

Saved 2673 examples.


In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

In [15]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [16]:
!pip install -U bitsandbytes>=0.46.1

In [28]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    device_map='cuda:0',
    trust_remote_code=True,
)
model.config.use_cache = False

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [29]:
from datasets import Dataset

In [30]:
SYSTEM = 'You are a historian analysing Titanic passenger records. Given a passenger profile, explain your reasoning and predict whether they survived.'

In [31]:
def format_example(ex):
    messages = [
        {'role': 'system',    'content': SYSTEM},
        {'role': 'user',      'content': ex['instruction']},
        {'role': 'assistant', 'content': ex['output']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

In [32]:
split    = int(len(examples) * 0.9)
train_ds = Dataset.from_list([format_example(e) for e in examples[:split]])
val_ds   = Dataset.from_list([format_example(e) for e in examples[split:]])
print(f"Train: {len(train_ds)}  Val: {len(val_ds)}")

Train: 2405  Val: 268


In [33]:
pip install -q transformers trl

Note: you may need to restart the kernel to use updated packages.


In [34]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

In [35]:
class PrintProgress(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Epoch {round(state.epoch,2)} | Step {state.global_step} | "
                  f"Loss {round(logs.get('loss',0),4)} | "
                  f"Eval Loss {round(logs.get('eval_loss',0),4) if 'eval_loss' in logs else '-'}")

In [36]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    args=SFTConfig(
        output_dir='output/lora',
        num_train_epochs=4,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        bf16=True,
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        report_to='none',
        max_length=512,
        dataset_text_field='text',
        packing=False,
    ),
    callbacks=[PrintProgress()],
)

Adding EOS to train dataset:   0%|          | 0/2405 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2405 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2405 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/268 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/268 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/268 [00:00<?, ? examples/s]

In [37]:
trainer.train()
trainer.model.save_pretrained('output/lora/final')
tokenizer.save_pretrained('output/lora/final')
print("Done. Adapter saved.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Epoch,Training Loss,Validation Loss
1,0.157862,0.163579
2,0.078191,0.086010
3,0.049782,0.057103
4,0.044762,0.055354


Epoch 0.07 | Step 10 | Loss 1.3296 | Eval Loss -
Epoch 0.13 | Step 20 | Loss 0.2933 | Eval Loss -
Epoch 0.2 | Step 30 | Loss 0.2777 | Eval Loss -
Epoch 0.27 | Step 40 | Loss 0.2595 | Eval Loss -
Epoch 0.33 | Step 50 | Loss 0.2405 | Eval Loss -
Epoch 0.4 | Step 60 | Loss 0.2332 | Eval Loss -
Epoch 0.47 | Step 70 | Loss 0.2215 | Eval Loss -
Epoch 0.53 | Step 80 | Loss 0.2177 | Eval Loss -
Epoch 0.6 | Step 90 | Loss 0.208 | Eval Loss -
Epoch 0.67 | Step 100 | Loss 0.1935 | Eval Loss -
Epoch 0.73 | Step 110 | Loss 0.201 | Eval Loss -
Epoch 0.8 | Step 120 | Loss 0.1866 | Eval Loss -
Epoch 0.86 | Step 130 | Loss 0.1745 | Eval Loss -
Epoch 0.93 | Step 140 | Loss 0.1684 | Eval Loss -
Epoch 1.0 | Step 150 | Loss 0.1579 | Eval Loss -
Epoch 1.0 | Step 151 | Loss 0 | Eval Loss 0.1636


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 1.06 | Step 160 | Loss 0.1326 | Eval Loss -
Epoch 1.13 | Step 170 | Loss 0.1325 | Eval Loss -
Epoch 1.19 | Step 180 | Loss 0.1285 | Eval Loss -
Epoch 1.26 | Step 190 | Loss 0.1272 | Eval Loss -
Epoch 1.33 | Step 200 | Loss 0.1159 | Eval Loss -
Epoch 1.39 | Step 210 | Loss 0.1128 | Eval Loss -
Epoch 1.46 | Step 220 | Loss 0.1061 | Eval Loss -
Epoch 1.53 | Step 230 | Loss 0.1011 | Eval Loss -
Epoch 1.59 | Step 240 | Loss 0.0986 | Eval Loss -
Epoch 1.66 | Step 250 | Loss 0.0963 | Eval Loss -
Epoch 1.72 | Step 260 | Loss 0.0986 | Eval Loss -
Epoch 1.79 | Step 270 | Loss 0.0882 | Eval Loss -
Epoch 1.86 | Step 280 | Loss 0.0813 | Eval Loss -
Epoch 1.92 | Step 290 | Loss 0.083 | Eval Loss -
Epoch 1.99 | Step 300 | Loss 0.0782 | Eval Loss -
Epoch 2.0 | Step 302 | Loss 0 | Eval Loss 0.086


...

Epoch 3.05 | Step 460 | Loss 0.0465 | Eval Loss -
Epoch 3.11 | Step 470 | Loss 0.0453 | Eval Loss -
Epoch 3.18 | Step 480 | Loss 0.0456 | Eval Loss -
Epoch 3.25 | Step 490 | Loss 0.0447 | Eval Loss -
Epoch 3.31 | Step 500 | Loss 0.0453 | Eval Loss -
Epoch 3.38 | Step 510 | Loss 0.0451 | Eval Loss -
Epoch 3.45 | Step 520 | Loss 0.0455 | Eval Loss -
Epoch 3.51 | Step 530 | Loss 0.0452 | Eval Loss -
Epoch 3.58 | Step 540 | Loss 0.0458 | Eval Loss -
Epoch 3.65 | Step 550 | Loss 0.0455 | Eval Loss -
Epoch 3.71 | Step 560 | Loss 0.0446 | Eval Loss -
Epoch 3.78 | Step 570 | Loss 0.0455 | Eval Loss -
Epoch 3.84 | Step 580 | Loss 0.0453 | Eval Loss -
Epoch 3.91 | Step 590 | Loss 0.0451 | Eval Loss -
Epoch 3.98 | Step 600 | Loss 0.0448 | Eval Loss -
Epoch 4.0 | Step 604 | Loss 0 | Eval Loss 0.0554


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 4.0 | Step 604 | Loss 0 | Eval Loss -


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Adapter saved.


In [38]:
def predict(passenger_description):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',   'content': f'Did this Titanic passenger survive?\n\n{passenger_description}'},
    ]
    inputs = tokenizer(
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
        return_tensors='pt'
    ).to(model.device)
 
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.3,
            do_sample=True,
            repetition_penalty=1.1,
        )
    print('=' * 55)
    print(passenger_description.strip())
    print('─' * 55)
    print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
    print()

In [39]:
predict("""
  - Name: Astor, Mr. John Jacob
  - Sex: male
  - Age: 47
  - Class: 1
  - Fare: £227.52
  - Siblings/Spouse aboard: 1
  - Parents/Children aboard: 0
  - Embarked from: Cherbourg
""")

- Name: Astor, Mr. John Jacob
  - Sex: male
  - Age: 47
  - Class: 1
  - Fare: £227.52
  - Siblings/Spouse aboard: 1
  - Parents/Children aboard: 0
  - Embarked from: Cherbourg
───────────────────────────────────────────────────────
Astor was an adult male passenger travelling first class, boarding at Cherbourg. Men had lower evacuation priority. As a first class passenger, they had priority access to lifeboats. Paid a high fare of £227.5. Travelled with 1 family member(s). Prediction: Did not survive.



In [40]:
predict("""
  - Name: Drew, Mrs. Lulu
  - Sex: female
  - Age: 34
  - Class: 2
  - Fare: £32.50
  - Siblings/Spouse aboard: 1
  - Parents/Children aboard: 1
  - Embarked from: Southampton
""")

- Name: Drew, Mrs. Lulu
  - Sex: female
  - Age: 34
  - Class: 2
  - Fare: £32.50
  - Siblings/Spouse aboard: 1
  - Parents/Children aboard: 1
  - Embarked from: Southampton
───────────────────────────────────────────────────────
Drew was an adult female passenger travelling second class, boarding at Southampton. Women were prioritised in the evacuation. As a second class passenger, they had moderate lifeboat access. Paid a moderate fare of £32.5. Travelled with 2 family member(s). Prediction: Survived.



In [41]:
predict("""
  - Name: Johnson, Mr. Erik
  - Sex: male
  - Age: 26
  - Class: 3
  - Fare: £7.78
  - Siblings/Spouse aboard: 0
  - Parents/Children aboard: 0
  - Embarked from: Southampton
""")

- Name: Johnson, Mr. Erik
  - Sex: male
  - Age: 26
  - Class: 3
  - Fare: £7.78
  - Siblings/Spouse aboard: 0
  - Parents/Children aboard: 0
  - Embarked from: Southampton
───────────────────────────────────────────────────────
Johnson was an adult male passenger travelling third class, boarding at Southampton. Men had lower evacuation priority. As a third class passenger, they had limited lifeboat access. Paid a low fare of £7.8. Travelled alone. Prediction: Did not survive.

